# Brugada-HUCA Synthetic ECG Generator (conditional WGAN-GP)

Notebook ini melatih GAN di Kaggle untuk membuat **5.000 pasien sintesis** dengan struktur folder output yang mengikuti dataset asli:

- `metadata.csv`
- `metadata_dictionary.csv`
- `RECORDS`
- `files/<patient_id>/<patient_id>.dat`
- `files/<patient_id>/<patient_id>.hea`

Silakan attach dataset Brugada-HUCA asli ke notebook Kaggle, atau aktifkan internet lalu set `download_if_missing=True`.


In [1]:
!pip install -q wfdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 12.1 MB/s eta 0:00:00


In [2]:
import os
import json
import math
import random
import shutil
import hashlib
import subprocess
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

try:
    import wfdb
except ImportError as exc:
    raise ImportError(
        "Package 'wfdb' belum terpasang. Di Kaggle, jalankan: !pip install -q wfdb"
    ) from exc


@dataclass
class Config:
    target_synthetic_patients: int = 5000
    seed: int = 42
    epochs: int = 350
    batch_size: int = 32
    lr: float = 2e-4
    betas: Tuple[float, float] = (0.5, 0.9)
    z_dim: int = 128
    cond_dim: int = 3
    embed_dim: int = 32
    base_channels: int = 64
    n_critic: int = 5
    gp_lambda: float = 10.0
    smooth_lambda: float = 1e-3
    num_workers: int = 0
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    input_root_env: str = "BRUGADA_DATASET_ROOT"
    input_root_default: str = "/kaggle/input/datasets/izzarsulynashrudin/brugada-huca"
    output_root: str = "/kaggle/working/brugada-huca-synthetic-gan"
    download_if_missing: bool = True
    original_download_url: str = "https://physionet.org/files/brugada-huca/1.0.0/"
    max_download_seconds: int = 600


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


class ECGDataset(Dataset):
    def __init__(self, signals: np.ndarray, conditions: np.ndarray):
        self.signals = torch.from_numpy(signals).float()
        self.conditions = torch.from_numpy(conditions).float()

    def __len__(self) -> int:
        return len(self.signals)

    def __getitem__(self, idx: int):
        return self.signals[idx], self.conditions[idx]


class ResidualBlock1D(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size=5, padding=2),
            nn.BatchNorm1d(channels),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(channels, channels, kernel_size=5, padding=2),
            nn.BatchNorm1d(channels),
        )
        self.act = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(x + self.net(x))


class Generator(nn.Module):
    def __init__(self, z_dim: int, cond_dim: int, embed_dim: int, out_channels: int = 12):
        super().__init__()
        self.cond_embed = nn.Sequential(
            nn.Linear(cond_dim, embed_dim),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(embed_dim, embed_dim),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.fc = nn.Linear(z_dim + embed_dim, 256 * 75)

        self.net = nn.Sequential(
            nn.Upsample(scale_factor=2, mode="linear", align_corners=False),
            nn.Conv1d(256, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.LeakyReLU(0.2, inplace=True),
            ResidualBlock1D(128),
            nn.Upsample(scale_factor=2, mode="linear", align_corners=False),
            nn.Conv1d(128, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(0.2, inplace=True),
            ResidualBlock1D(64),
            nn.Upsample(scale_factor=2, mode="linear", align_corners=False),
            nn.Conv1d(64, 32, kernel_size=5, padding=2),
            nn.BatchNorm1d(32),
            nn.LeakyReLU(0.2, inplace=True),
            ResidualBlock1D(32),
            nn.Upsample(scale_factor=2, mode="linear", align_corners=False),
            nn.Conv1d(32, 16, kernel_size=5, padding=2),
            nn.BatchNorm1d(16),
            nn.LeakyReLU(0.2, inplace=True),
            ResidualBlock1D(16),
            nn.Conv1d(16, out_channels, kernel_size=7, padding=3),
        )

    def forward(self, z: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:
        cond_emb = self.cond_embed(cond)
        x = torch.cat([z, cond_emb], dim=1)
        x = self.fc(x).view(-1, 256, 75)
        x = self.net(x)
        return x


class Critic(nn.Module):
    def __init__(self, in_channels: int = 12, cond_dim: int = 3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_channels + cond_dim, 32, kernel_size=5, stride=2, padding=2),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(32, 64, kernel_size=5, stride=2, padding=2),
            nn.InstanceNorm1d(64, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(64, 128, kernel_size=5, stride=2, padding=2),
            nn.InstanceNorm1d(128, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv1d(128, 256, kernel_size=5, stride=2, padding=2),
            nn.InstanceNorm1d(256, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.head = nn.Linear(256 * 75, 1)

    def forward(self, x: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:
        cond_map = cond.unsqueeze(-1).repeat(1, 1, x.shape[-1])
        x = torch.cat([x, cond_map], dim=1)
        x = self.net(x)
        x = x.flatten(1)
        return self.head(x)


class Standardizer:
    def __init__(self, mean: np.ndarray, std: np.ndarray, mins: np.ndarray, maxs: np.ndarray):
        self.mean = mean.astype(np.float32)
        self.std = std.astype(np.float32)
        self.mins = mins.astype(np.float32)
        self.maxs = maxs.astype(np.float32)

    @classmethod
    def fit(cls, x: np.ndarray) -> "Standardizer":
        mean = x.mean(axis=(0, 2), keepdims=True)
        std = x.std(axis=(0, 2), keepdims=True) + 1e-6
        mins = x.min(axis=(0, 2), keepdims=True)
        maxs = x.max(axis=(0, 2), keepdims=True)
        return cls(mean, std, mins, maxs)

    def transform(self, x: np.ndarray) -> np.ndarray:
        return (x - self.mean) / self.std

    def inverse_transform(self, x: np.ndarray) -> np.ndarray:
        restored = (x * self.std) + self.mean
        lower = self.mins - 0.15 * np.abs(self.mins)
        upper = self.maxs + 0.15 * np.abs(self.maxs)
        return np.clip(restored, lower, upper)



def locate_dataset_root(config: Config) -> Path:
    explicit_root = os.getenv(config.input_root_env)
    candidate_roots: List[Path] = []

    if explicit_root:
        candidate_roots.append(Path(explicit_root))

    candidate_roots.extend(
        [
            Path(config.input_root_default),
            Path(config.input_root_default) / "physionet.org" / "files" / "brugada-huca" / "1.0.0",
            Path(config.input_root_default) / "brugada-huca" / "1.0.0",
        ]
    )

    search_roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path("/kaggle/temp")]
    candidate_roots.extend(search_roots)
    for base in search_roots:
        if not base.exists():
            continue
        candidate_roots.extend(
            [
                base / "physionet.org" / "files" / "brugada-huca" / "1.0.0",
                base / "files" / "brugada-huca" / "1.0.0",
            ]
        )

    seen = set()
    for candidate in candidate_roots:
        candidate = candidate.resolve() if candidate.exists() else candidate
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if (candidate / "metadata.csv").exists():
            if (candidate / "files").exists() or (candidate / "files" / "files").exists():
                return candidate

    for base in search_roots + [Path(config.input_root_default)]:
        if not base.exists():
            continue
        for metadata_path in base.rglob("metadata.csv"):
            candidate = metadata_path.parent
            if (candidate / "files").exists() or (candidate / "files" / "files").exists():
                return candidate

    if config.download_if_missing:
        download_root = Path("/kaggle/working/brugada-huca-original")
        download_root.mkdir(parents=True, exist_ok=True)
        try:
            subprocess.run(
                [
                    "bash",
                    "-lc",
                    "set -e; cd /kaggle/working; rm -rf brugada-huca-original; mkdir -p brugada-huca-original; cd brugada-huca-original; wget -q -r -N -c -np --cut-dirs=2 -R 'index.html*' https://physionet.org/files/brugada-huca/1.0.0/",
                ],
                check=True,
                timeout=config.max_download_seconds,
            )
            for metadata_path in download_root.rglob("metadata.csv"):
                candidate = metadata_path.parent
                if (candidate / "files").exists() or (candidate / "files" / "files").exists():
                    return candidate
        except Exception as exc:
            print(f"Download otomatis gagal: {exc}")

    raise FileNotFoundError(
        "Dataset root tidak ditemukan. Pastikan dataset asli sudah ditambahkan ke Kaggle Input, "
        "atau set env BRUGADA_DATASET_ROOT ke folder yang berisi metadata.csv dan files/."
    )


def resolve_record_path(dataset_root: Path, patient_id: str) -> Path:
    patient_id = str(patient_id).replace(".0", "")
    candidates = [
        dataset_root / "files" / patient_id / patient_id,
        dataset_root / "files" / "files" / patient_id / patient_id,
        dataset_root / patient_id / patient_id,
    ]
    for record_path in candidates:
        if record_path.with_suffix(".hea").exists() and record_path.with_suffix(".dat").exists():
            return record_path
    raise FileNotFoundError(
        "Record tidak lengkap untuk patient_id="
        f"{patient_id}. Dicoba di: " + " | ".join(str(p) for p in candidates)
    )


def load_original_dataset(dataset_root: Path) -> Tuple[pd.DataFrame, np.ndarray, Dict[str, object]]:
    metadata = pd.read_csv(dataset_root / "metadata.csv")
    required_cols = ["patient_id", "basal_pattern", "sudden_death", "brugada"]
    missing = [c for c in required_cols if c not in metadata.columns]
    if missing:
        raise ValueError(f"Kolom metadata wajib tidak ditemukan: {missing}")

    metadata["patient_id"] = (
        metadata["patient_id"]
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
    )

    signals: List[np.ndarray] = []
    template = None
    valid_rows: List[int] = []

    for row_idx, patient_id in enumerate(metadata["patient_id"].tolist()):
        try:
            record_path = resolve_record_path(dataset_root, patient_id)
        except FileNotFoundError:
            print(f"Warning: record patient_id={patient_id} tidak ditemukan, dilewati.")
            continue

        record = wfdb.rdrecord(str(record_path))
        if record.p_signal.shape != (1200, 12):
            raise ValueError(
                f"Dimensi sinyal tak sesuai untuk patient_id={patient_id}: {record.p_signal.shape}, ekspektasi (1200, 12)"
            )
        signals.append(record.p_signal.T.astype(np.float32))
        valid_rows.append(row_idx)
        if template is None:
            template = {
                "fs": float(record.fs),
                "sig_name": list(record.sig_name),
                "units": list(record.units) if getattr(record, "units", None) else ["mV"] * record.n_sig,
                "fmt": ["16"] * record.n_sig,
            }

    if not signals:
        raise RuntimeError("Tidak ada record yang berhasil dibaca dari dataset asli.")

    metadata = metadata.iloc[valid_rows].reset_index(drop=True)
    signal_array = np.stack(signals, axis=0)
    return metadata, signal_array, template


def compute_gradient_penalty(
    critic: Critic,
    real_samples: torch.Tensor,
    fake_samples: torch.Tensor,
    cond: torch.Tensor,
    device: str,
) -> torch.Tensor:
    alpha = torch.rand(real_samples.size(0), 1, 1, device=device)
    interpolates = (alpha * real_samples + (1 - alpha) * fake_samples).requires_grad_(True)
    critic_scores = critic(interpolates, cond)
    grad_outputs = torch.ones_like(critic_scores, device=device)
    gradients = torch.autograd.grad(
        outputs=critic_scores,
        inputs=interpolates,
        grad_outputs=grad_outputs,
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]
    gradients = gradients.view(gradients.size(0), -1)
    return ((gradients.norm(2, dim=1) - 1) ** 2).mean()


def train_cwgan_gp(
    signals: np.ndarray,
    metadata: pd.DataFrame,
    config: Config,
) -> Tuple[Generator, Standardizer, Dict[str, List[float]]]:
    conditions = metadata[["basal_pattern", "sudden_death", "brugada"]].astype(np.float32).values
    standardizer = Standardizer.fit(signals)
    signals_norm = standardizer.transform(signals)

    dataset = ECGDataset(signals_norm, conditions)
    loader = DataLoader(
        dataset,
        batch_size=min(config.batch_size, len(dataset)),
        shuffle=True,
        num_workers=config.num_workers,
        pin_memory=(config.device == "cuda"),
        drop_last=False,
    )

    generator = Generator(config.z_dim, config.cond_dim, config.embed_dim).to(config.device)
    critic = Critic(in_channels=signals.shape[1], cond_dim=config.cond_dim).to(config.device)

    opt_g = optim.Adam(generator.parameters(), lr=config.lr, betas=config.betas)
    opt_d = optim.Adam(critic.parameters(), lr=config.lr, betas=config.betas)

    history = {"critic_loss": [], "generator_loss": []}
    step = 0

    for epoch in range(1, config.epochs + 1):
        g_epoch, d_epoch = [], []
        for real_batch, cond_batch in loader:
            real_batch = real_batch.to(config.device, non_blocking=True)
            cond_batch = cond_batch.to(config.device, non_blocking=True)
            batch_size = real_batch.size(0)

            z = torch.randn(batch_size, config.z_dim, device=config.device)
            fake_batch = generator(z, cond_batch)
            real_score = critic(real_batch, cond_batch).mean()
            fake_score = critic(fake_batch.detach(), cond_batch).mean()
            gp = compute_gradient_penalty(critic, real_batch, fake_batch.detach(), cond_batch, config.device)
            d_loss = fake_score - real_score + config.gp_lambda * gp

            opt_d.zero_grad(set_to_none=True)
            d_loss.backward()
            opt_d.step()
            d_epoch.append(float(d_loss.item()))

            if step % config.n_critic == 0:
                z = torch.randn(batch_size, config.z_dim, device=config.device)
                fake_batch = generator(z, cond_batch)
                gen_score = critic(fake_batch, cond_batch).mean()
                smoothness = torch.mean(torch.abs(fake_batch[:, :, 1:] - fake_batch[:, :, :-1]))
                g_loss = -gen_score + config.smooth_lambda * smoothness

                opt_g.zero_grad(set_to_none=True)
                g_loss.backward()
                opt_g.step()
                g_epoch.append(float(g_loss.item()))

            step += 1

        history["critic_loss"].append(float(np.mean(d_epoch)))
        history["generator_loss"].append(float(np.mean(g_epoch) if g_epoch else np.nan))

        if epoch == 1 or epoch % max(1, config.epochs // 10) == 0:
            print(
                f"Epoch {epoch:04d}/{config.epochs} | "
                f"D={history['critic_loss'][-1]:.4f} | G={history['generator_loss'][-1]:.4f}"
            )

    return generator, standardizer, history


def sample_condition_vectors(metadata: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    cond_df = metadata[["basal_pattern", "sudden_death", "brugada"]].astype(int)
    combo_counts = cond_df.value_counts(normalize=True).reset_index(name="prob")
    combos = combo_counts[["basal_pattern", "sudden_death", "brugada"]].values
    probs = combo_counts["prob"].values

    rng = np.random.default_rng(seed)
    chosen_idx = rng.choice(len(combos), size=n, p=probs)
    sampled = combos[chosen_idx]
    return pd.DataFrame(sampled, columns=["basal_pattern", "sudden_death", "brugada"]).astype(int)


@torch.no_grad()
def generate_synthetic_signals(
    generator: Generator,
    standardizer: Standardizer,
    synthetic_conditions: pd.DataFrame,
    config: Config,
) -> np.ndarray:
    generator.eval()
    device = config.device
    outputs = []
    batch_size = 128

    cond_arr = synthetic_conditions[["basal_pattern", "sudden_death", "brugada"]].astype(np.float32).values
    for start in range(0, len(cond_arr), batch_size):
        cond_batch = torch.from_numpy(cond_arr[start:start + batch_size]).float().to(device)
        z = torch.randn(cond_batch.size(0), config.z_dim, device=device)
        fake = generator(z, cond_batch).cpu().numpy()
        outputs.append(fake)

    generated = np.concatenate(outputs, axis=0)
    generated = standardizer.inverse_transform(generated)
    return generated.astype(np.float32)


def prepare_output_root(output_root: Path) -> None:
    if output_root.exists():
        shutil.rmtree(output_root)
    (output_root / "files").mkdir(parents=True, exist_ok=True)


def create_metadata_dictionary(output_root: Path, original_root: Path) -> None:
    src = original_root / "metadata_dictionary.csv"
    dst = output_root / "metadata_dictionary.csv"
    if src.exists():
        shutil.copy2(src, dst)
    else:
        pd.DataFrame(
            {
                "variable": ["patient_id", "basal_pattern", "sudden_death", "brugada"],
                "description": [
                    "Unique synthetic subject identifier",
                    "Baseline ECG exhibits pathological Brugada-type pattern (0/1)",
                    "Synthetic sudden death label (0/1)",
                    "Synthetic Brugada diagnosis label (0/1)",
                ],
            }
        ).to_csv(dst, index=False)


def write_synthetic_dataset(
    synthetic_signals: np.ndarray,
    synthetic_metadata: pd.DataFrame,
    template: Dict[str, object],
    original_root: Path,
    output_root: Path,
    config: Config,
) -> None:
    prepare_output_root(output_root)
    create_metadata_dictionary(output_root, original_root)

    readme_text = f"""# Brugada-HUCA Synthetic ECG Dataset (GAN)

Dataset ini berisi data sintesis yang dihasilkan dari conditional WGAN-GP menggunakan dataset Brugada-HUCA asli.

- Jumlah pasien sintesis: {len(synthetic_metadata)}
- Panjang rekaman: 12 detik
- Sampling frequency: {template['fs']} Hz
- Jumlah lead: {len(template['sig_name'])}
- Struktur direktori mengikuti dataset asli:
  - metadata.csv
  - metadata_dictionary.csv
  - RECORDS
  - files/<patient_id>/<patient_id>.dat
  - files/<patient_id>/<patient_id>.hea

Catatan:
- Ini adalah data sintesis untuk eksperimen machine learning, bukan pengganti data klinis asli.
- Distribusi label metadata disampling dari frekuensi kombinasi label pada dataset asli.
"""
    (output_root / "README.md").write_text(readme_text, encoding="utf-8")

    license_src = original_root / "LICENSE.txt"
    if license_src.exists():
        shutil.copy2(license_src, output_root / "LICENSE.txt")

    synthetic_metadata = synthetic_metadata.copy()
    synthetic_metadata.to_csv(output_root / "metadata.csv", index=False)

    record_lines: List[str] = []
    for row_idx, row in synthetic_metadata.iterrows():
        patient_id = str(int(row["patient_id"]))
        patient_dir = output_root / "files" / patient_id
        patient_dir.mkdir(parents=True, exist_ok=True)

        signal = synthetic_signals[row_idx].T  # (1200, 12)
        wfdb.wrsamp(
            record_name=patient_id,
            fs=template["fs"],
            units=template["units"],
            sig_name=template["sig_name"],
            p_signal=signal,
            fmt=template["fmt"],
            write_dir=str(patient_dir),
        )
        record_lines.append(f"files/{patient_id}/{patient_id}")

    (output_root / "RECORDS").write_text("\n".join(record_lines) + "\n", encoding="utf-8")

    config_payload = asdict(config)
    (output_root / "SYNTHESIS_CONFIG.json").write_text(
        json.dumps(config_payload, indent=2), encoding="utf-8"
    )

    training_note = {
        "model": "conditional WGAN-GP",
        "condition_columns": ["basal_pattern", "sudden_death", "brugada"],
        "shape_per_record": [12, 1200],
        "output_root": str(output_root),
    }
    (output_root / "SYNTHESIS_INFO.json").write_text(
        json.dumps(training_note, indent=2), encoding="utf-8"
    )


def write_sha256_sums(output_root: Path) -> None:
    sha_lines = []
    for path in sorted(output_root.rglob("*")):
        if path.is_file() and path.name != "SHA256SUMS.txt":
            hasher = hashlib.sha256()
            with open(path, "rb") as f:
                for chunk in iter(lambda: f.read(1024 * 1024), b""):
                    hasher.update(chunk)
            rel_path = path.relative_to(output_root)
            sha_lines.append(f"{hasher.hexdigest()}  {rel_path.as_posix()}")
    (output_root / "SHA256SUMS.txt").write_text("\n".join(sha_lines) + "\n", encoding="utf-8")


def zip_output(output_root: Path) -> Path:
    zip_base = output_root.parent / output_root.name
    zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=str(output_root))
    return Path(zip_path)


def assign_new_patient_ids(original_metadata: pd.DataFrame, n: int) -> np.ndarray:
    max_id = int(original_metadata["patient_id"].astype(int).max())
    return np.arange(max_id + 1, max_id + 1 + n, dtype=np.int64)


def main() -> None:
    config = Config()
    seed_everything(config.seed)

    print("[1/6] Mencari dataset asli...")
    dataset_root = locate_dataset_root(config)
    print(f"Dataset root: {dataset_root}")

    print("[2/6] Membaca ECG dan metadata...")
    original_metadata, original_signals, template = load_original_dataset(dataset_root)
    print(f"Loaded {len(original_metadata)} pasien dengan shape sinyal {original_signals.shape}.")

    print("[3/6] Melatih conditional WGAN-GP...")
    generator, standardizer, history = train_cwgan_gp(original_signals, original_metadata, config)

    output_root = Path(config.output_root)

    print("[4/6] Menyusun metadata sintesis...")
    synthetic_metadata = sample_condition_vectors(
        original_metadata,
        n=config.target_synthetic_patients,
        seed=config.seed + 7,
    )
    synthetic_metadata.insert(0, "patient_id", assign_new_patient_ids(original_metadata, len(synthetic_metadata)))

    print("[5/6] Menghasilkan sinyal sintesis...")
    synthetic_signals = generate_synthetic_signals(generator, standardizer, synthetic_metadata, config)

    print("[6/6] Menulis dataset WFDB sintesis...")
    write_synthetic_dataset(
        synthetic_signals=synthetic_signals,
        synthetic_metadata=synthetic_metadata,
        template=template,
        original_root=dataset_root,
        output_root=output_root,
        config=config,
    )
    torch.save(generator.state_dict(), output_root / "generator_weights.pt")
    (output_root / "training_history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
    write_sha256_sums(output_root)
    zip_path = zip_output(output_root)

    print("Selesai.")
    print(f"Output folder  : {output_root}")
    print(f"Output zip     : {zip_path}")
    print(f"Metadata synth : {output_root / 'metadata.csv'}")


if __name__ == "__main__":
    main()


[1/6] Mencari dataset asli...
Dataset root: /kaggle/input/datasets/izzarsulynashrudin/brugada-huca
[2/6] Membaca ECG dan metadata...
Loaded 363 pasien dengan shape sinyal (363, 12, 1200).
[3/6] Melatih conditional WGAN-GP...


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:330.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch 0001/350 | D=0.3594 | G=0.8778
Epoch 0035/350 | D=-47.3046 | G=53.9776
Epoch 0070/350 | D=-52.2357 | G=69.8315
Epoch 0105/350 | D=-63.5076 | G=78.9581
Epoch 0140/350 | D=-71.6547 | G=89.1418
Epoch 0175/350 | D=-79.4465 | G=100.7506
Epoch 0210/350 | D=-87.4538 | G=86.8683
Epoch 0245/350 | D=-83.9719 | G=78.0366
Epoch 0280/350 | D=-80.9459 | G=92.6963
Epoch 0315/350 | D=-87.2938 | G=68.4596
Epoch 0350/350 | D=-87.4194 | G=87.9028
[4/6] Menyusun metadata sintesis...
[5/6] Menghasilkan sinyal sintesis...
[6/6] Menulis dataset WFDB sintesis...
Selesai.
Output folder  : /kaggle/working/brugada-huca-synthetic-gan
Output zip     : /kaggle/working/brugada-huca-synthetic-gan.zip
Metadata synth : /kaggle/working/brugada-huca-synthetic-gan/metadata.csv


In [3]:
# Jalankan pipeline end-to-end
main()


[1/6] Mencari dataset asli...
Dataset root: /kaggle/input/datasets/izzarsulynashrudin/brugada-huca
[2/6] Membaca ECG dan metadata...
Loaded 363 pasien dengan shape sinyal (363, 12, 1200).
[3/6] Melatih conditional WGAN-GP...
Epoch 0001/350 | D=0.3597 | G=0.8777
Epoch 0035/350 | D=-45.2010 | G=51.8688
Epoch 0070/350 | D=-49.0562 | G=66.8690
Epoch 0105/350 | D=-65.3599 | G=81.3687
Epoch 0140/350 | D=-70.9528 | G=86.8458
Epoch 0175/350 | D=-76.0621 | G=86.3111
Epoch 0210/350 | D=-85.1122 | G=84.5701
Epoch 0245/350 | D=-88.3166 | G=71.9006
Epoch 0280/350 | D=-97.0872 | G=105.5549
Epoch 0315/350 | D=-101.5282 | G=92.8568
Epoch 0350/350 | D=-94.8015 | G=81.9918
[4/6] Menyusun metadata sintesis...
[5/6] Menghasilkan sinyal sintesis...
[6/6] Menulis dataset WFDB sintesis...
Selesai.
Output folder  : /kaggle/working/brugada-huca-synthetic-gan
Output zip     : /kaggle/working/brugada-huca-synthetic-gan.zip
Metadata synth : /kaggle/working/brugada-huca-synthetic-gan/metadata.csv


In [4]:
# Cek struktur output
from pathlib import Path
output_root = Path("/kaggle/working/brugada-huca-synthetic-gan")
for path in [output_root, output_root / "files", output_root / "metadata.csv", output_root / "RECORDS", output_root / "generator_weights.pt"]:
    print(path, "exists=", path.exists())
print("Contoh folder pasien:")
if (output_root / "files").exists():
    sample_dirs = sorted([p for p in (output_root / "files").iterdir() if p.is_dir()])[:3]
    for p in sample_dirs:
        print(" -", p)


/kaggle/working/brugada-huca-synthetic-gan exists= True
/kaggle/working/brugada-huca-synthetic-gan/files exists= True
/kaggle/working/brugada-huca-synthetic-gan/metadata.csv exists= True
/kaggle/working/brugada-huca-synthetic-gan/RECORDS exists= True
/kaggle/working/brugada-huca-synthetic-gan/generator_weights.pt exists= True
Contoh folder pasien:
 - /kaggle/working/brugada-huca-synthetic-gan/files/3139197
 - /kaggle/working/brugada-huca-synthetic-gan/files/3139198
 - /kaggle/working/brugada-huca-synthetic-gan/files/3139199
